# HW – Part 3: Generate the written report
Run this **after** notebooks 1 and 2 (same Colab session, or upload the `outputs/` folder).
It reads the saved metrics and writes `outputs/REPORT.md` (+ `REPORT.docx` if pandoc is available)
with tables, figures and an analysis built from your actual numbers.

In [ ]:
import json, os, subprocess
import pandas as pd

GITHUB_USER = "YOUR_GITHUB_USERNAME"   # <-- edit
REPO        = "ml-onnx-deploy"         # <-- edit (repo name you created)
STUDENT     = "Hoang Thien Thanh Le (Tea)"
COURSE      = "ITS 52000 – Applied Machine Learning, Purdue University Northwest"
PAGES = f"https://{GITHUB_USER}.github.io/{REPO}"

R = json.load(open("outputs/results/regression_metrics.json"))
C = json.load(open("outputs/results/classification_metrics.json"))
rs = pd.DataFrame(R["summary"])
cs = pd.DataFrame(C["summary"])

def f(x, d=3):
    return f"{x:.{d}f}"

In [ ]:
# ---------- regression narrative ----------
lin = rs[rs.model == "Linear"].iloc[0]
best_r = rs.loc[rs.test_R2_mean.idxmax()]
dep_r = rs[rs.model == R["best_model"]].iloc[0]
abl = pd.DataFrame(R["ablation"])
def abl_val(model, feat):
    return abl[(abl.model == model) & (abl["age feature"] == feat)].test_R2_mean.iloc[0]
lin_raw, lin_log = abl_val("Linear", "raw age"), abl_val("Linear", "log(1+age)")
mlp_raw, mlp_log = abl_val("MLP", "raw age"), abl_val("MLP", "log(1+age)")
imp_r = list(R["permutation_importance"].items())
top_r = ", ".join(k for k, _ in imp_r[:3])
most_stable = rs.loc[rs.test_R2_std.idxmin(), "model"]
least_stable = rs.loc[rs.test_R2_std.idxmax(), "model"]

reg_table = "| Model | Val R² | Test R² (mean ± std) | Test MAE (MPa) | Test RMSE (MPa) | Epochs |\n|---|---|---|---|---|---|\n"
for _, r in rs.iterrows():
    star = " ★" if r.model == R["best_model"] else ""
    reg_table += (f"| {r.model}{star} | {f(r.val_R2)} | {f(r.test_R2_mean)} ± {f(r.test_R2_std)} | "
                  f"{f(r.test_MAE, 2)} | {f(r.test_RMSE, 2)} | {r.epochs:.0f} |\n")
abl_table = "| Model | raw age | log(1+age) |\n|---|---|---|\n"
for m in ["Linear", "MLP"]:
    abl_table += f"| {m} | {f(abl_val(m, 'raw age'))} | {f(abl_val(m, 'log(1+age)'))} |\n"

# ---------- classification narrative ----------
dep_c = cs[cs.model == C["best_model"]].iloc[0]
best_c = cs.loc[cs.F1.idxmax()]
logreg = cs[cs.model == "Logistic Regression"].iloc[0]
cm = C["confusion_matrix"]; tn, fp, fn, tp = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
m, mt = C["test_metrics"], C["test_metrics_tuned"]
imp_c = list(C["permutation_importance"].items())
top_c = ", ".join(k for k, _ in imp_c[:3])
corr_c = sorted(C["target_corr"].items(), key=lambda kv: -abs(kv[1]))
gap_lr = best_c.F1 - logreg.F1
pos = C["class_counts"]["1"]; neg = C["class_counts"]["0"]

gap_r = best_r.test_R2_mean - lin.test_R2_mean
if best_r.model == "Linear" or gap_r < 0.02:
    reg_nonlin = (f"* **Linear vs non-linear.** The linear model (with log-age) already reaches R² = {f(lin.test_R2_mean)}; "
                  f"the best network ({best_r.model}, {f(best_r.test_R2_mean)}) adds only {f(gap_r)}. With good feature "
                  f"engineering most of the signal is captured by a linear relation, and extra capacity mainly adds variance.")
else:
    reg_nonlin = (f"* **Non-linearity matters.** The linear model explains {f(lin.test_R2_mean)} of the variance, while the best "
                  f"non-linear model ({best_r.model}) reaches {f(best_r.test_R2_mean)} — an improvement of {f(gap_r)} in R². "
                  f"Strength depends on interactions such as the water/cement ratio and on a saturating ageing curve, "
                  f"which a linear model cannot represent.")

if best_c.model == "Logistic Regression" or gap_lr < 0.03:
    clf_capacity = (f"* **Model capacity.** The best mean test F1 is {f(best_c.F1)} ({best_c.model}) versus {f(logreg.F1)} for "
                    f"logistic regression, a difference of {f(gap_lr)}. On a small, noisy dataset with mostly monotonic risk "
                    f"factors, extra depth brings limited benefit and adds variance across seeds; data quality (about half "
                    f"of the insulin values are imputed) limits performance more than the architecture does.")
else:
    clf_capacity = (f"* **Model capacity.** The best mean test F1 is {f(best_c.F1)} ({best_c.model}) versus {f(logreg.F1)} for "
                    f"logistic regression, an improvement of {f(gap_lr)}, suggesting some non-linear interactions between "
                    f"risk factors (e.g. glucose with BMI and age). The ± std column shows whether this gain is larger than "
                    f"the seed-to-seed variation.")

clf_table = "| Model | Val F1 | Test F1 (mean ± std) | Precision | Recall | Accuracy | MCC | ROC-AUC |\n|---|---|---|---|---|---|---|---|\n"
for _, r in cs.iterrows():
    star = " ★" if r.model == C["best_model"] else ""
    clf_table += (f"| {r.model}{star} | {f(r.val_F1)} | {f(r.F1)} ± {f(r.F1_std)} | {f(r.precision)} | "
                  f"{f(r.recall)} | {f(r.accuracy)} | {f(r.MCC)} | {f(r.ROC_AUC)} |\n")

In [ ]:
report = f"""# Regression and Classification on Tabular Data with PyTorch, ONNX and GitHub Pages

**Student:** {STUDENT}
**Course:** {COURSE}
**Live demo:** {PAGES}/
**Code:** https://github.com/{GITHUB_USER}/{REPO}

## 1. Overview

Two complete machine-learning pipelines were built on public tabular datasets: a **regression** model that predicts
concrete compressive strength and a **classification** model that predicts diabetes. For each task five PyTorch
architectures discussed in class were trained with the same loop and compared — a linear model, a one-hidden-layer
MLP, a deeper network with dropout, the **F1+F2** (linear + non-linear) model and a **residual network**. Each
architecture was trained with five random seeds and the deployed model was selected on a held-out validation set,
so the test set is used only for the final report. Following the class deployment example, all pre-processing
statistics are stored inside the model, which is exported with `torch.onnx.export` (opset 15, `input1` → `output1`)
and executed in the browser with onnxruntime-web on GitHub Pages.

## 2. Part 1 — Regression: Concrete Compressive Strength

### 2.1 Data
The Concrete Compressive Strength dataset (Yeh, 1998; UCI repository) has {R['n_rows_raw']} laboratory samples with
8 numeric inputs — cement, blast-furnace slag, fly ash, water, superplasticizer, coarse and fine aggregate
(kg/m³) and age (days) — and the target compressive strength in MPa. There are no missing values;
{R['n_duplicates_removed']} duplicated rows were removed so that identical samples cannot leak between the training
and test sets. The data were split 70/15/15 into training ({R['n_train']}), validation ({R['n_val']}) and test
({R['n_test']}) sets.

![Correlation matrix](figures/reg_correlation.png)

Cement (r = {f(R['target_corr']['cement'], 2)}), superplasticizer (r = {f(R['target_corr']['superplastic'], 2)})
and age (r = {f(R['target_corr']['age'], 2)}) correlate positively with strength, while water is negative
(r = {f(R['target_corr']['water'], 2)}), reflecting the well-known water-to-cement ratio effect.

### 2.2 Processing
* **Feature engineering:** age is right-skewed (1–365 days) and strength gain saturates after about 28 days, so
  `log(1 + age)` is used (figure below).
* **Standardization:** inputs and target are z-scored with training-set means and standard deviations; the network
  predicts the scaled target and the output is de-scaled to MPa.
* Both steps are implemented inside `forward()`, so the ONNX file accepts raw values.

![Age transform](figures/reg_age_log.png)

### 2.3 Models and training
Loss MSE on the scaled target, Adam (lr 1e-3, weight decay 1e-5), batch 32, up to 2 000 epochs with early stopping
on validation loss (patience 150) and restoration of the best weights.

### 2.4 Results

{reg_table}
★ = deployed model (highest mean validation R²).

![Model comparison](figures/reg_model_comparison.png)

![Diagnostics of the best model](figures/reg_best_model_diagnostics.png)

The deployed **{R['best_model']}** reaches **R² = {f(R['test_metrics']['R2'])}** on the test set, with
MAE = {f(R['test_metrics']['MAE'], 2)} MPa and RMSE = {f(R['test_metrics']['RMSE'], 2)} MPa.

### 2.5 Analysis
{reg_nonlin}
* **Feature engineering is a cheap win.** Replacing raw age with `log(1+age)` raised the linear model from
  R² = {f(lin_raw)} to {f(lin_log)} and the MLP from {f(mlp_raw)} to {f(mlp_log)}. Domain knowledge narrows the gap
  between simple and complex models.

{abl_table}
* **Stability.** Over five seeds the {most_stable} model was the most stable and {least_stable} the least stable
  (largest R² standard deviation), so the ranking of models that differ by less than one standard deviation should
  not be over-interpreted.
* **Error pattern.** In the predicted-vs-actual plot the points lie close to the diagonal; the residual plot shows
  whether errors widen for the strongest mixes, where training examples are rarer.
* **Feature importance.** Permutation importance ranks **{top_r}** as the most influential inputs, consistent with
  concrete engineering: strength is driven mainly by curing time and binder content.

![Permutation importance](figures/reg_permutation_importance.png)

## 3. Part 2 — Classification: Pima Indians Diabetes

### 3.1 Data
768 female patients of Pima heritage aged ≥ 21 years, 8 clinical inputs (pregnancies, 2-h plasma glucose,
diastolic blood pressure, triceps skin-fold thickness, 2-h serum insulin, BMI, diabetes pedigree function, age) and a
binary outcome (diabetes within five years). The classes are imbalanced: {neg} negative vs {pos} positive
({f(pos / (pos + neg) * 100, 1)} % positive). A stratified 70/15/15 split gave {C['n_train']} / {C['n_val']} /
{C['n_test']} samples.

**Hidden missing values.** A value of 0 is physiologically impossible for glucose, blood pressure, skin thickness,
insulin and BMI. Zeros appear {C['zeros']['insulin']} times for insulin and {C['zeros']['skin_thickness']} times for
skin thickness, so they were treated as missing.

![Distributions by class](figures/clf_distributions.png)

### 3.2 Processing
* Zeros in those five columns are replaced by the **median of the non-zero training values**.
* Inputs are standardized with training-set statistics.
* The loss is **class-weighted cross-entropy** to counter the imbalance.
* Imputation and scaling are embedded in the model (`torch.where` + buffers), so the ONNX graph reproduces them in
  the browser; the exported model outputs softmax probabilities.

![Correlation matrix](figures/clf_correlation.png)

The correlation matrix shows that the outcome is most correlated with
{", ".join(f"{k} (r = {v:.2f})" for k, v in corr_c[:3])}. Among the inputs, age–pregnancies and BMI–skin thickness are
the strongest pairs, but no pair is so highly correlated that a feature had to be dropped.

### 3.3 Models and training
Adam (lr 1e-3, weight decay 1e-4), batch 32, up to 1 000 epochs, early stopping on validation loss (patience 60),
five seeds per architecture.

### 3.4 Results

{clf_table}
★ = deployed model (highest mean validation F1). F1, precision and recall refer to the diabetes class at threshold 0.5.

![Results](figures/clf_results.png)

The deployed **{C['best_model']}** obtains on the test set: **F1 = {f(m['F1'])}**, precision = {f(m['precision'])},
recall = {f(m['recall'])}, accuracy = {f(m['accuracy'])}, MCC = {f(m['MCC'])} and ROC-AUC = {f(m['ROC_AUC'])}.
Confusion matrix: TN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}.

### 3.5 Analysis
* **Accuracy is not enough.** Because 65 % of patients are negative, a model that always predicts "no diabetes"
  would already reach about 65 % accuracy with F1 = 0. F1, recall, MCC and AUC are therefore the main metrics.
{clf_capacity}
* **Errors.** The model misses {fn} of {fn + tp} diabetic patients in the test set (false negatives) and raises
  {fp} false alarms. For screening, missing a diabetic patient is usually more costly than a false alarm.
* **Threshold.** A threshold of {C['threshold_tuned']} chosen on the validation set changes test precision/recall
  from {f(m['precision'])}/{f(m['recall'])} to {f(mt['precision'])}/{f(mt['recall'])} (F1 {f(m['F1'])} →
  {f(mt['F1'])}). The web demo shows both decisions.
* **Feature importance.** Permutation importance ranks **{top_c}** highest, matching clinical knowledge that plasma
  glucose and adiposity are the main diabetes risk factors.

![Threshold](figures/clf_threshold.png)
![Permutation importance](figures/clf_permutation_importance.png)

## 4. Deployment with ONNX and GitHub Pages
1. The selected model is wrapped so that it returns a single tensor (MPa for regression; softmax probabilities for
   classification) and exported with `torch.onnx.export(..., input_names=["input1"], output_names=["output1"],
   opset_version=15, dynamic_axes=...)`, as in the class example.
2. The ONNX file is checked with onnxruntime in Python: the maximum difference to the PyTorch output is printed in the
   notebooks.
3. Static pages (`index.html` + `index.js`) load `onnxruntime-web`, build a `float32` tensor of shape `[1, 8]` from
   the form and call `session.run({{ input1: tensor }})`. A `metrics.json` file shows the evaluation results next to the
   prediction.
4. The `docs/` folder is published with GitHub Pages: {PAGES}/regression/ and {PAGES}/classification/.

## 5. Conclusion
For concrete strength, non-linear networks clearly outperform linear regression, and a domain-informed transform of
age gives a large improvement at no cost. For diabetes prediction, the gain from deeper networks is small; careful
handling of hidden missing values, class imbalance and the decision threshold matter more than architecture.
Embedding the pre-processing inside the PyTorch module makes the ONNX models self-contained, so both run fully in the
browser without a server.

## References
* Yeh, I-C. (1998). Modeling of strength of high-performance concrete using artificial neural networks. *Cement and
  Concrete Research*, 28(12), 1797–1808.
* Smith, J. W., Everhart, J. E., Dickson, W. C., Knowler, W. C., & Johannes, R. S. (1988). Using the ADAP learning
  algorithm to forecast the onset of diabetes mellitus. *Proc. Symposium on Computer Applications in Medical Care*, 261–265.
* Calix, R. A. ConstraintsMLprediction – ONNX deployment examples. https://github.com/rcalix1/ConstraintsMLprediction
* PyTorch ONNX export documentation; ONNX Runtime Web documentation.
"""
open("outputs/REPORT.md", "w", encoding="utf-8").write(report)
print("Wrote outputs/REPORT.md")

In [ ]:
# Optional: Word version (pandoc is pre-installed on Colab)
try:
    subprocess.run(["pandoc", "REPORT.md", "-o", "REPORT.docx", "--resource-path=."], cwd="outputs", check=True)
    print("Wrote outputs/REPORT.docx")
except Exception as e:
    print("pandoc not available – open REPORT.md instead.", e)

In [ ]:
import shutil
shutil.make_archive("ALL_outputs", "zip", "outputs")
try:
    from google.colab import files; files.download("ALL_outputs.zip")
except Exception:
    print("Saved ALL_outputs.zip")